In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model=ChatGroq(model="openai/gpt-oss-120b")

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False
            }
        )
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [14]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='bb20c9dd-d931-4de5-91c5-ef84f900174d'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send an email to team@company.com about the Q4 results. The user didn\'t specify subject or body. Probably we need to ask for more details? The instruction: "Send an email to team@company.com about the Q4 results". We could assume a generic subject and body? Could ask clarification, but likely they want us to send an email. We need subject and body. We can default: Subject: "Q4 Results Overview". Body: "Dear team, Please find the Q4 results attached..." But we have no attachment. Could just write a brief note.\n\nWe can ask for clarification: "What should the email contain?" But maybe they expect us to send a simple email.\n\nI\'ll send an email with a generic message.', 'tool_calls': [{'i

In [15]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The email titled **“Q4 Results Summary”** has been sent to team@company.com. If you need any revisions or additional follow‑up, just let me know!


In [17]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions":[{"type":"reject","reason":"Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I’m ready to run that deletion, but it’s a permanent change that can’t be undone. Would you like me to go ahead and delete all rows from the **users** table where `active = false`? Please confirm if you’d like me to proceed.
